# Sprint 1 — BluaDiagnostics PoC

PoC funcional com Ollama Cloud, system prompt clínico, memória conversacional e function calling simulado.

In [ ]:
!pip install -q ollama python-dotenv ipython


In [ ]:
import os, json
from ollama import Client
from IPython.display import Markdown, display

try:
    from google.colab import userdata
    OLLAMA_API_KEY = userdata.get("OLLAMA_API_KEY")
    print("✅ Ambiente: Google Colab")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    OLLAMA_API_KEY = os.getenv("OLLAMA_API_KEY")
    print("✅ Ambiente: Local/VSCode")

if not OLLAMA_API_KEY:
    raise ValueError("Configure OLLAMA_API_KEY no Colab Secrets ou no arquivo .env")

client = Client(host="https://ollama.com", headers={"Authorization": "Bearer " + OLLAMA_API_KEY})
MODEL_NAME = "gpt-oss:120b"


## System prompt aplicado

In [ ]:
system_prompt = """
PAPEL:
Você é o BluaDiagnostics, assistente conversacional da Care Plus para check-up digital.

ESCOPO:
Coletar sintomas, sinais vitais informados, contexto básico e apoiar encaminhamento seguro.

RESTRIÇÕES:
Não diagnosticar, não prescrever, não substituir médico, respeitar LGPD e minimizar dados sensíveis.

FORMATO_DE_SAIDA:
Responder em português com: 1. Resumo do relato 2. Pontos de atenção 3. Próxima ação 4. Aviso médico.

ESCALADA_HUMANA:
Em dor no peito, falta de ar intensa, desmaio, confusão mental ou sinais neurológicos, orientar atendimento imediato.
"""
display(Markdown(system_prompt))


## Dados mockados e function calling

In [ ]:
paciente_mock = {
    "paciente_id": "PAC001",
    "idade": 42,
    "condicoes": ["hipertensão leve"],
    "alergias": ["dipirona"],
    "medicamentos": ["losartana 50mg"],
    "wearable_mock": {"pressao": "135/85", "frequencia_cardiaca": 82, "saturacao": 98}
}

def consultar_historico_paciente(paciente_id):
    print("🔧 Tool chamada: consultar_historico_paciente")
    return paciente_mock if paciente_id == "PAC001" else {"erro": "paciente não encontrado"}

def verificar_interacoes_medicamentosas(medicamentos):
    print("🔧 Tool chamada: verificar_interacoes_medicamentosas")
    meds = [m.lower() for m in medicamentos]
    if "dipirona" in meds:
        return {"risco": "alto", "mensagem": "Paciente possui alergia registrada a dipirona."}
    if "ibuprofeno" in meds and any("losartana" in m for m in meds):
        return {"risco": "moderado", "mensagem": "Atenção ao uso de anti-inflamatório em paciente hipertenso."}
    return {"risco": "baixo", "mensagem": "Sem interação relevante na base simulada."}

def agendar_teleconsulta(paciente_id, especialidade, prioridade):
    print("🔧 Tool chamada: agendar_teleconsulta")
    return {"status": "agendado", "paciente_id": paciente_id, "especialidade": especialidade, "prioridade": prioridade, "data_hora": "2026-05-20 14:30"}


## Memória conversacional com 3 turnos

In [ ]:
memoria = [{"role": "system", "content": system_prompt}]

def chamar_llm(mensagem_usuario):
    memoria.append({"role": "user", "content": mensagem_usuario})
    resposta = client.chat(model=MODEL_NAME, messages=memoria, options={"temperature": 0.3, "num_predict": 300}, stream=False)
    texto = resposta["message"]["content"]
    memoria.append({"role": "assistant", "content": texto})
    display(Markdown("### Resposta do BluaDiagnostics\n" + texto))
    return texto

chamar_llm("Olá, quero fazer um check-up digital. Tenho 42 anos e sou hipertenso.")
chamar_llm("Hoje acordei com dor de cabeça leve e pressão 135 por 85.")
chamar_llm("Também senti tontura ao levantar rápido.")


## Execução das tools

In [ ]:
historico = consultar_historico_paciente("PAC001")
interacoes = verificar_interacoes_medicamentosas(["losartana", "ibuprofeno"])
teleconsulta = agendar_teleconsulta("PAC001", "Clínico Geral", "moderada")

print(json.dumps({"historico": historico, "interacoes": interacoes, "teleconsulta": teleconsulta}, indent=2, ensure_ascii=False))


## Resposta final com contexto das tools

In [ ]:
contexto = f"""
Histórico: {historico}
Interações: {interacoes}
Teleconsulta: {teleconsulta}

Gere uma orientação segura para o beneficiário. Não diagnostique e não prescreva.
"""
chamar_llm(contexto)


## Considerações técnicas

A PoC demonstra system prompt clínico, memória conversacional, function calling simulado, integração com Ollama Cloud e guardrails clínicos. A arquitetura foi preparada para evolução com RAG e orquestração multi-agente nas próximas sprints.